# Runtime probe & benchmark — Llama 3.1 8B sinh mã Python

Notebook này là **điểm vào duy nhất** để chạy thí nghiệm trên GPU ngoài (Colab/Kaggle). Nó cài dependency, đồng bộ repo, kiểm tra runtime, xác thực quyền HF, chạy smoke test, chạy benchmark A/B và đóng gói artifact.

| Mục | Giá trị |
|---|---|
| Model | `meta-llama/Llama-3.1-8B-Instruct` @ `0e9e39f249a16976918f6564b8830bc894c89659` |
| Backend | `local` 4-bit trên CUDA (HF Inference Providers không có provider mapping cho checkpoint này) |
| Yêu cầu | Runtime có GPU ≥ 8 GB (khuyến nghị T4 16 GB); HF token đã chấp nhận license |
| Đầu ra | `results/raw/benchmark_*.jsonl`, `results/summary/benchmark_*.json`, `results_export_<run_id>.zip` |

**Thứ tự chạy:** 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8. Mỗi cell có assertion; nếu cell trước chưa đạt, đừng chạy cell sau. Output được xóa trước khi commit để không lộ token/tên tài khoản/log tải trọng số.


In [ ]:
# 1. Cấu hình dùng chung cho toàn notebook.
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/datkony/llama-code-generation.git"
REPO_NAME = "llama-code-generation"
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
REVISION = "0e9e39f249a16976918f6564b8830bc894c89659"
VARIANTS = "A,B"
PYTHON = sys.executable
WORKDIR = Path("/content") if Path("/content").is_dir() else Path.cwd()


def run(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    """Chạy lệnh, in log giới hạn và trả về process để parse output."""
    print("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.stderr:
        print(proc.stderr[-2000:], file=sys.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (exit={proc.returncode}): {' '.join(cmd)}")
    return proc


print("Python", sys.version.split()[0], "| workdir", WORKDIR)

In [ ]:
# 2. Dependency. Gỡ diffusers vì bản Colab đòi huggingface-hub>=1.32, xung đột transformers 4.x.
run([PYTHON, "-m", "pip", "uninstall", "-y", "diffusers"], check=False)
run(
    [
        PYTHON, "-m", "pip", "install", "-q",
        "transformers>=4.49,<5",
        "accelerate>=1.3,<2",
        "huggingface_hub>=0.30,<1",
        "bitsandbytes>=0.45,<1",
        "gradio>=5,<6",
        "PyYAML>=6,<7",
    ]
)

import torch
import transformers
import huggingface_hub

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__, "| hub", huggingface_hub.__version__)

In [ ]:
# 3. Đồng bộ mã nguồn. Luôn quay về WORKDIR để tránh clone lồng nhau.
os.chdir(WORKDIR)
repo_dir = WORKDIR / REPO_NAME
nested_dir = repo_dir / REPO_NAME  # bản clone lồng từ các lần chạy cũ (nếu có)

if (repo_dir / ".git").is_dir():
    run(["git", "-C", str(repo_dir), "pull", "--ff-only"])
elif (nested_dir / ".git").is_dir():
    print("Cảnh báo: phát hiện bản clone lồng nhau; dùng thư mục trong cùng.")
    repo_dir = nested_dir
    run(["git", "-C", str(repo_dir), "pull", "--ff-only"])
else:
    run(["git", "clone", "-q", REPO_URL, REPO_NAME])

os.chdir(repo_dir)
assert (Path.cwd() / "scripts" / "check_runtime.py").exists(), "Repo chưa được đồng bộ đúng."
print("Repo sẵn sàng:", Path.cwd())

In [ ]:
# 4. Kiểm tra runtime. Dừng ngay nếu không có CUDA để tránh rơi vào endpoint không hỗ trợ.
proc = run([PYTHON, "scripts/check_runtime.py"])
runtime = json.loads(proc.stdout)

print(
    json.dumps(
        {
            "runtime": runtime["runtime"],
            "python": runtime["python"],
            "cuda_available": runtime["torch"]["cuda_available"],
            "gpu": runtime["torch"]["gpu"],
            "vram_mib": runtime["torch"]["vram_mib"],
            "recommended_backend": runtime["recommended_backend"],
            "keep_8b": runtime["keep_8b"],
        },
        ensure_ascii=False,
        indent=2,
    )
)
assert runtime["torch"]["cuda_available"], (
    "Không có CUDA. Vào Runtime → Change runtime type → T4 GPU, rồi chạy lại từ cell 2."
)
assert runtime["keep_8b"], "VRAM không đủ cho 8B; dừng để nhóm quyết định fallback."

In [ ]:
# 5. Xác thực quyền HF. Token chỉ nằm trong biến môi trường của kernel.
from getpass import getpass

from huggingface_hub import HfApi, hf_hub_download

token = os.environ.get("HF_TOKEN", "").strip()
if not token:
    print("Nhập HF token đã chấp nhận license Llama 3.1 (không ghi vào notebook).")
    try:
        token = getpass("HF token (ẩn): ").strip()
    except Exception:
        token = ""
    if not token:
        token = input("HF token (input): ").strip()

if not token:
    raise RuntimeError("Cần HF token để tải checkpoint gated.")

os.environ["HF_TOKEN"] = token
account = (HfApi(token=token).whoami().get("name") or "").strip()
print("HF account:", account[:2] + "***" if account else "<unknown>")
config_path = hf_hub_download(
    repo_id=MODEL_ID, filename="config.json", revision=REVISION, token=token
)
print("Gated access OK:", Path(config_path).name)

# In các trường kiến trúc đọc trực tiếp từ checkpoint gated (không suy từ mirror).
with open(config_path, encoding="utf-8") as fh:
    model_config = json.load(fh)
CONFIG_FIELDS = (
    "architectures", "num_hidden_layers", "hidden_size", "intermediate_size",
    "num_attention_heads", "num_key_value_heads", "head_dim", "vocab_size",
    "max_position_embeddings", "rope_theta", "rope_scaling", "rms_norm_eps",
    "hidden_act", "torch_dtype", "use_cache",
)
print("--- Checkpoint config ---")
for key in CONFIG_FIELDS:
    print(f"{key} = {model_config.get(key, '<missing>')}")
token = ""  # xóa giá trị thô ngay sau khi dùng


In [ ]:
# 6. Smoke test đầu-cuối: prompt → generation → trích code → ast → sandbox test.
proc = run([PYTHON, "scripts/smoke_test.py", "--backend", "local"], check=False)
if proc.returncode != 0:
    raise RuntimeError("Smoke test chưa đạt; không chạy benchmark.")
print("Smoke test: PASS")

In [ ]:
# 7. Benchmark chính thức A/B trên tập report + kiểm thử đơn vị.
proc = run(
    [PYTHON, "scripts/run_benchmark.py", "--backend", "local", "--variants", VARIANTS],
    check=False,
)
if proc.returncode != 0:
    raise RuntimeError("Benchmark thất bại; giữ lại log và báo nhóm.")

summary, _ = json.JSONDecoder().raw_decode(proc.stdout[proc.stdout.index("{"):])
assert summary["offline"] is False, "Kết quả offline không được báo cáo."
assert summary["model"]["model_id"] == MODEL_ID
expected = summary["dataset"]["n_tasks"] * len(VARIANTS.split(","))
assert summary["n_records"] == expected, "Số bản ghi không khớp thiết kế thí nghiệm."

for variant, metrics in summary["variants"].items():
    print(
        f"{variant}: pass@1={metrics['pass_at_1']:.3f} | "
        f"syntax={metrics['syntax_validity']:.3f} | "
        f"latency_mean={metrics['latency_mean_s']:.3f}s | "
        f"errors={metrics['error_counts']}"
    )

tests = run([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-q"], check=False)
if tests.returncode != 0:
    raise RuntimeError("Unit tests thất bại sau benchmark.")
print("Unit tests: PASS")

In [ ]:
# 8. Kiểm tra artifact và đóng gói để mang về máy local.
from zipfile import ZipFile

raw_files = sorted(Path("results/raw").glob("benchmark_*.jsonl"))
summary_files = sorted(Path("results/summary").glob("benchmark_*.json"))
assert raw_files and summary_files, "Thiếu file kết quả."

zip_path = Path(f"results_export_{summary['run_id']}.zip")
with ZipFile(zip_path, "w") as bundle:
    for path in raw_files + summary_files:
        bundle.write(path)

print("raw:", *(p.name for p in raw_files))
print("summary:", *(p.name for p in summary_files))
print("Gói kết quả:", zip_path.resolve())

## Ghi chú vận hành

- **Không dùng Inference Providers** cho checkpoint này: `inferenceProviderMapping` trống, mọi thử `provider=...` đều báo model không được hỗ trợ.
- Cần chấp nhận license tại https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct bằng đúng tài khoản tạo token.
- `--offline` chỉ validate pipeline; số liệu của nó không được đưa vào báo cáo.
- Nếu đổi runtime/GPU, chạy lại từ cell 2; Colab restart sẽ mất package cài ở cell 1.
- Trước khi commit, xóa output notebook (`Edit → Clear All Outputs`) để tránh lộ thông tin.
